# Notebook 05 - Faithfulness of the explanations: Objective (c), RQ2/H2
Thesis Section 3.5.3, Table 3.2 and **Equation 3**.

| Test | What a faithful method shows |
|---|---|
| Deletion and insertion (Eq. 3) | output falls faster on deletion, rises faster on insertion, than a random order |
| Randomisation checks | maps change a lot when the model's weights are randomised or labels shuffled |
| Stability | maps stay similar under tiny noise that does not change the prediction |
| Cross-method agreement | methods that pass the other tests agree with each other |
| Known-answer tests | attribution lands on the planted segment, or on the QRS for a QRS-duration model |

Runs on a stratified sample of 500 test records (100 per superclass). GPU recommended; about 1-3 hours.

In [ ]:
# STEP 0 - Setup. Run this cell first, every time you open the notebook.
import os, sys, glob, json, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # the branch of your GitHub repository to use

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                       # click "Connect to Google Drive" and allow
    PROJECT = "/content/drive/MyDrive/FUTA-MTECH"        # everything you keep lives here
    REPO = "/content/FUTA-MTECH-PROJECT"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/Scientist47/FUTA-MTECH-PROJECT.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip -q install wfdb neurokit2 shap captum interpret-core
else:                                                    # running on your own computer
    PROJECT = os.environ.get("FUTA_PROJECT", os.path.abspath(".."))
    REPO = os.environ.get("FUTA_REPO", os.path.abspath(".."))

sys.path.insert(0, REPO)
os.chdir(REPO)
DATA = f"{PROJECT}/data"            # downloaded and processed data
RESULTS = f"{PROJECT}/results"      # tables, figures and trained models for Chapter 4
for d in (DATA, RESULTS, f"{RESULTS}/tables", f"{RESULTS}/figures", f"{RESULTS}/models"):
    os.makedirs(d, exist_ok=True)

# QUICK_TEST = True runs everything with tiny settings, to check the notebook works.
# Leave it False for your real thesis results.
QUICK_TEST = os.environ.get("FUTA_QUICK_TEST") == "1"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
print("GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only - see the guide, Step 1.3)")
print("Project folder:", PROJECT)

In [ ]:
from scipy.stats import wilcoxon
from preprocessing.ptbxl import SUPERCLASSES, LEADS
from training.train import load_trained, train
from models import build_model
from xai.shap_explainer import explain_shap
from xai.gradcam import explain_gradcam
from xai.attention import extract_attention
from xai.segment_alignment import align_batch, batch_boundaries, SEGMENTS
from evaluation import faithfulness as F
P = f"{DATA}/processed/ptbxl100"
X_test, y_test = np.load(f"{P}/X_test.npy"), np.load(f"{P}/y_test.npy")
X_train = np.load(f"{P}/X_train.npy", mmap_mode="r")
masks = np.load(f"{RESULTS}/segment_masks_test.npy")
store = dict(np.load(f"{RESULTS}/grids_test.npz"))
models = {a: load_trained(a, 5, f"{RESULTS}/models/ptbxl_{a}/seed0")[0] for a in ["cnn1d", "cnn_bilstm_attention"]}
rng = np.random.default_rng(0)
background = np.asarray(X_train[np.sort(rng.choice(len(X_train), 20 if QUICK_TEST else 100, replace=False))])
COMBOS = [("shap", "cnn1d"), ("gradcam", "cnn1d"), ("shap", "cnn_bilstm_attention"),
          ("gradcam", "cnn_bilstm_attention"), ("attention", "cnn_bilstm_attention")]
PER_CLASS = 2 if QUICK_TEST else 100       # 5 x 100 = 500 records
STEP = 6 if QUICK_TEST else 1              # cells removed per deletion step (1 = one cell at a time, K = 60)

## Step 5.1 - The stratified sample of 500 test records

In [ ]:
sample = []   # (row in X_test, class index, position inside that class's grid array)
for c, cls in enumerate(SUPERCLASSES):
    idx = store[f"idx__{cls}"]
    usable = np.where((masks[idx] >= 0).mean(axis=1) > 0.2)[0]    # records with delineated beats
    pos = np.sort(rng.choice(usable, min(PER_CLASS, len(usable)), replace=False))
    sample += [(int(idx[p]), c, int(p)) for p in pos]
rows_ = np.array([s[0] for s in sample]); cls_ = np.array([s[1] for s in sample])
Xs, Ms = X_test[rows_], masks[rows_]
def grids_for(method, arch):
    return np.stack([store[f"{method}__{arch}__{SUPERCLASSES[c]}"][p] for _, c, p in sample])
print(len(sample), "record-class pairs")

## Step 5.2 - Deletion and insertion AUC (Equation 3) vs the random-attribution baseline
For each record the 60 cells are removed (set to 0, the training mean) from most to least important, and the model's probability for the class is recorded after each step. A random ordering of the cells (a random attribution map) is the baseline.

In [ ]:
res, curves = [], {}
for method, arch in COMBOS:
    g = grids_for(method, arch)
    d = F.deletion_auc(models[arch], Xs, Ms, g, cls_, step=STEP)
    ins = F.insertion_auc(models[arch], Xs, Ms, g, cls_, step=STEP)
    pd_ = wilcoxon(d["explanation"], d["random"]).pvalue
    pi_ = wilcoxon(ins["explanation"], ins["random"]).pvalue
    res.append({"Method": method, "Model": arch,
                "Deletion AUC (method)": d["explanation"].mean(), "Deletion AUC (random)": d["random"].mean(),
                "Deletion p": pd_,
                "Insertion AUC (method)": ins["explanation"].mean(), "Insertion AUC (random)": ins["random"].mean(),
                "Insertion p": pi_,
                "Passes": bool(d["explanation"].mean() < d["random"].mean() and pd_ < 0.05
                               and ins["explanation"].mean() > ins["random"].mean() and pi_ < 0.05)})
    curves[(method, arch)] = np.mean([F.deletion_curve(models[arch], x, m, gg, c, STEP)
                                      for x, m, gg, c in zip(Xs[:50], Ms[:50], g[:50], cls_[:50])], axis=0)
    print(method, arch, "done")
table47 = pd.DataFrame(res).round(4)
table47.to_csv(f"{RESULTS}/tables/table4_7_deletion_insertion.csv", index=False); table47

In [ ]:
# Figure 4.6: mean deletion curves (first 50 sampled records), with the random baseline
plt.figure(figsize=(7, 4.5))
for (method, arch), cv in curves.items():
    plt.plot(np.arange(len(cv)) * STEP, cv, label=f"{method} - {arch}")
rc = np.mean([F.deletion_curve(models["cnn1d"], x, m, gg, c, STEP, rng=np.random.default_rng(1))
              for x, m, gg, c in zip(Xs[:50], Ms[:50], grids_for("shap", "cnn1d")[:50], cls_[:50])], axis=0)
plt.plot(np.arange(len(rc)) * STEP, rc, "k--", label="random order (cnn1d)")
plt.xlabel("cells removed"); plt.ylabel("predicted probability for the class"); plt.legend(fontsize=8)
plt.title("Deletion curves"); plt.tight_layout()
plt.savefig(f"{RESULTS}/figures/fig4_6_deletion_curves.png", dpi=300); plt.show()

## Step 5.3 - Randomisation checks (Adebayo et al., 2018)
(a) **Cascading layer randomisation:** layers are re-initialised from the output backwards and the maps recomputed. (b) **Shuffled labels:** a 1D-CNN is retrained on randomly shuffled training labels. In both, a faithful method's maps should become **unlike** the original (low Spearman correlation).

In [ ]:
sub = slice(0, 10 if QUICK_TEST else 100)
Xr, Mr, cr = Xs[sub], Ms[sub], cls_[sub]
def explain_fn_factory(method):
    def fn(model, X):
        out = []
        for c in np.unique(cr):
            sel = cr == c
            if method == "shap":
                maps = explain_shap(model, X[sel], background, int(c), nsamples=10 if QUICK_TEST else 50)
            elif method == "gradcam":
                maps = explain_gradcam(model, X[sel], int(c))
            else:
                maps = extract_attention(model, X[sel])
            out.append((np.where(sel)[0], align_batch(maps, Mr[sel])))
        g = np.zeros((len(X), 5, 12), dtype=np.float32)
        for where, gg in out:
            g[where] = gg
        return g
    return fn
rand_rows = []
for method, arch in COMBOS:
    print(f"--- {method}, {arch}")
    r = F.randomisation_check(models[arch], explain_fn_factory(method), Xr, Mr)
    rand_rows.append({"Method": method, "Model": arch, "Spearman after top layer": r[0][1],
                      "Spearman, all layers randomised": r[-1][1]})
table48a = pd.DataFrame(rand_rows).round(3); table48a

In [ ]:
# (b) shuffled-label model (one seed of the 1D-CNN, fewer epochs)
import yaml
cfg = yaml.safe_load(open("training/configs/ptbxl_cnn1d.yaml"))
cfg.update(max_epochs=2 if QUICK_TEST else 15, patience=3)
shuffled = {"X_train": np.asarray(X_train), "y_train": np.load(f"{P}/y_train.npy").copy(),
            "X_val": np.load(f"{P}/X_val.npy"), "y_val": np.load(f"{P}/y_val.npy")}
shuffled["y_train"] = shuffled["y_train"][np.random.default_rng(0).permutation(len(shuffled["y_train"]))]
sdir = f"{RESULTS}/models/ptbxl_cnn1d_shuffled"
if os.path.exists(f"{sdir}/seed0/best.pt"):
    m_shuf = load_trained("cnn1d", 5, f"{sdir}/seed0")[0]
else:
    m_shuf, _ = train(cfg, 0, shuffled, out_dir=sdir)
del shuffled
shuf_rows = []
for method in ["shap", "gradcam"]:
    fn = explain_fn_factory(method)
    sim = F.grid_similarity(fn(models["cnn1d"], Xr), fn(m_shuf, Xr)).mean()
    shuf_rows.append({"Method": method, "Model": "cnn1d", "Spearman, shuffled-label model": round(float(sim), 3)})
table48b = pd.DataFrame(shuf_rows)
table48 = table48a.merge(table48b, how="left", on=["Method", "Model"])
table48.to_csv(f"{RESULTS}/tables/table4_8_randomisation.csv", index=False); table48

## Step 5.4 - Stability under small noise

In [ ]:
stab_rows = []
for method, arch in COMBOS:
    # small Gaussian noise (0.02 SD in z-score units); keep records whose prediction is unchanged
    fn = explain_fn_factory(method)
    noisy = (Xr + 0.02 * np.random.default_rng(1).standard_normal(Xr.shape)).astype(np.float32)
    with torch.no_grad():
        dev = next(models[arch].parameters()).device
        p0 = torch.sigmoid(models[arch](torch.as_tensor(Xr, device=dev))).cpu().numpy() >= 0.5
        p1 = torch.sigmoid(models[arch](torch.as_tensor(noisy, device=dev))).cpu().numpy() >= 0.5
    keep = (p0 == p1).all(axis=1)
    sims = F.grid_similarity(fn(models[arch], Xr), fn(models[arch], noisy))[keep]
    stab_rows.append({"Method": method, "Model": arch, "Records with unchanged prediction": int(keep.sum()),
                      "Mean Spearman": round(float(sims.mean()), 3)})
table49 = pd.DataFrame(stab_rows)
table49.to_csv(f"{RESULTS}/tables/table4_9_stability.csv", index=False); table49

## Step 5.5 - Cross-method agreement (interpret only for methods that passed the tests above)

In [ ]:
agree = []
pairs = [(("shap", "cnn1d"), ("gradcam", "cnn1d")),
         (("shap", "cnn_bilstm_attention"), ("gradcam", "cnn_bilstm_attention")),
         (("shap", "cnn_bilstm_attention"), ("attention", "cnn_bilstm_attention")),
         (("gradcam", "cnn_bilstm_attention"), ("attention", "cnn_bilstm_attention")),
         (("shap", "cnn1d"), ("shap", "cnn_bilstm_attention"))]
for a, b in pairs:
    r = F.method_agreement(grids_for(*a), grids_for(*b), k=10)
    agree.append({"A": " / ".join(a), "B": " / ".join(b), "Spearman": round(r["spearman_mean"], 3),
                  "Top-10 overlap": round(r["topk_overlap_mean"], 3)})
table410 = pd.DataFrame(agree)
table410.to_csv(f"{RESULTS}/tables/table4_10_agreement.csv", index=False); table410

## Step 5.6 - Known-answer tests
**(a) Planted waveform.** A smooth bump is added to the ST segment of half of a set of training records, and a fresh 1D-CNN learns to detect it. A faithful method should put most of its attribution on the ST segment.

**(b) QRS-duration regression.** A 1D-CNN learns to predict QRS duration (from PTB-XL+ where available). A faithful method should concentrate on the QRS complex.

In [ ]:
n_ka = 300 if QUICK_TEST else 4000
Xk = np.asarray(X_train[:n_ka]); Mk = batch_boundaries(Xk, verbose=False)
Xp, yp = F.plant_waveform(Xk, Mk, segment=3, amplitude=1.0)
cut = int(0.8 * len(Xp))
dev = "cuda" if torch.cuda.is_available() else "cpu"
m_plant = F.fit_quick(build_model("cnn1d", 12, 1).to(dev), Xp[:cut], yp[:cut], epochs=2 if QUICK_TEST else 8)
pos = np.where(yp[cut:, 0] == 1)[0][:100] + cut
ka_rows = []
for method in ["shap", "gradcam"]:
    maps = explain_shap(m_plant, Xp[pos], Xp[:50], 0, nsamples=10 if QUICK_TEST else 50) if method == "shap" \
        else explain_gradcam(m_plant, Xp[pos], 0)
    share = F.segment_share(align_batch(maps, Mk[pos]), 3)
    ka_rows.append({"Test": "planted waveform (ST)", "Method": method, "Share on target segment": round(float(share.mean()), 3)})
pd.DataFrame(ka_rows)

In [ ]:
# (b) QRS-duration regression
paths = json.load(open(f"{PROJECT}/paths.json"))
qrs = None
if "PTBXL_PLUS" in paths and os.path.exists(paths["PTBXL_PLUS"]):
    from models.ebm_comparator import load_ptbxl_plus_features
    ids_tr = np.load(f"{P}/ecg_id_train.npy")[:n_ka]
    f = load_ptbxl_plus_features(paths["PTBXL_PLUS"], ids_tr)
    cand = [c for c in f.columns if "qrs" in c.lower() and "dur" in c.lower()]
    print("QRS duration columns found:", cand[:5])
    if cand:
        qrs = f[cand[0]].to_numpy(dtype=float)
if qrs is None:        # fallback: QRS duration from our own delineation (state this in Chapter 4)
    qrs = np.array([np.mean(np.diff(np.where(np.diff(np.r_[0, (m == 2).astype(int), 0]) != 0)[0])[::2]) if (m == 2).any() else np.nan for m in Mk])
ok = ~np.isnan(qrs)
target = ((qrs - np.nanmean(qrs)) / np.nanstd(qrs))[:, None].astype(np.float32)
okidx = np.where(ok)[0]; cut_q = int(0.8 * len(okidx))
m_qrs = F.fit_quick(build_model("cnn1d", 12, 1).to(dev), Xk[okidx[:cut_q]], target[okidx[:cut_q]], loss="mse",
                    epochs=2 if QUICK_TEST else 8)
test_idx = okidx[cut_q:cut_q + 100]
for method in ["shap", "gradcam"]:
    maps = explain_shap(m_qrs, Xk[test_idx], Xk[:50], 0, nsamples=10 if QUICK_TEST else 50) if method == "shap" \
        else explain_gradcam(m_qrs, Xk[test_idx], 0)
    share = F.segment_share(align_batch(maps, Mk[test_idx]), 2)
    ka_rows.append({"Test": "QRS-duration regression (QRS)", "Method": method, "Share on target segment": round(float(share.mean()), 3)})
seg_len = np.array([[(m == s).mean() for s in range(5)] for m in Mk]).mean(0)
print("For reference, share of samples in each segment (what a uniform map would give):",
      dict(zip(SEGMENTS, seg_len.round(3))))
table411 = pd.DataFrame(ka_rows)
table411.to_csv(f"{RESULTS}/tables/table4_11_known_answer.csv", index=False); table411

**Reading the known-answer table:** the share on the target segment should be well above the segment's share of samples printed above (what a uniform map would give).

**Done.** For Chapter 4, Section 4.5 keep Tables 4.7-4.11 and Figure 4.6. Next: notebook 06 (clinician study).